# 🏈 Fantasy Football Data Analysis Pipeline

This notebook provides a complete analysis workflow for your fantasy football league data stored in Cloudflare R2.

## Pipeline Overview

1. **Setup & Explore Data** - Load projections, injuries, rosters, and stats from R2
2. **Analysis Workflows** - Lineup optimizer, injury reports, performance tracking, trade analyzer
3. **Save Results to R2** - Export analysis results back to R2 for frontend consumption
4. **Integration Guide** - How your Vite app can consume the results

---

## Data Sources (R2 Bucket)

* `fantasai/projections/season=2026/current_week.json` - Player projections (423 KB)
* `fantasai/injuries/latest.json` - Injury status (132 KB)
* `fantasai/rosters/latest.json` - Team rosters (0.32 KB)
* `fantasai/stats/season=2026/week=1/data.json` - Actual performance stats

---

**Let's get started! 🚀**

In [0]:
import requests
import pandas as pd
import json
import io
from pyspark.sql import functions as F
from pyspark.sql.window import Window

class R2Proxy:
    """Access Cloudflare R2 through the fantasai-api Worker proxy."""

    def __init__(self, base_url="https://api.fantasai.net", api_key=None):
        self.base = base_url.rstrip("/")
        self.headers = {}
        if api_key:
            self.headers["X-FantasAI-Key"] = api_key

    def list_objects(self, prefix=""):
        resp = requests.get(
            f"{self.base}/api/v1/r2/list",
            params={"prefix": prefix},
            headers=self.headers,
        )
        resp.raise_for_status()
        return resp.json()["objects"]

    def download(self, key):
        """Download an object. Returns bytes."""
        resp = requests.get(
            f"{self.base}/api/v1/r2/{key}",
            headers=self.headers,
        )
        resp.raise_for_status()
        return resp.content

    def upload(self, key, data, content_type="application/octet-stream"):
        """Upload bytes or a file-like object."""
        resp = requests.put(
            f"{self.base}/api/v1/r2/{key}",
            data=data,
            headers={**self.headers, "Content-Type": content_type},
        )
        resp.raise_for_status()
        return resp.json()

    def delete(self, key):
        resp = requests.delete(
            f"{self.base}/api/v1/r2/{key}",
            headers=self.headers,
        )
        resp.raise_for_status()
        return resp.json()

    def download_json_serverless(self, key):
        """Download JSON from R2 and return as Spark DataFrame (serverless-compatible)."""
        data = self.download(key)
        parsed = json.loads(data.decode('utf-8'))
        
        # Convert to pandas first, then to Spark (works on serverless)
        if isinstance(parsed, list):
            pandas_df = pd.DataFrame(parsed)
        else:
            pandas_df = pd.DataFrame([parsed])
        
        return spark.createDataFrame(pandas_df)
    
    def upload_dataframe_serverless(self, df, key, fmt="csv"):
        """Upload a Spark DataFrame to R2 (serverless-compatible)."""
        # Convert to Pandas (works on serverless)
        pandas_df = df.toPandas()
        
        # Serialize to bytes
        buffer = io.BytesIO()
        if fmt == "csv":
            pandas_df.to_csv(buffer, index=False)
            content_type = "text/csv"
        elif fmt == "json":
            pandas_df.to_json(buffer, orient="records", lines=True)
            content_type = "application/json"
        else:
            raise ValueError(f"Unsupported format: {fmt}")
        
        # Upload to R2
        buffer.seek(0)
        result = self.upload(key, buffer.getvalue(), content_type=content_type)
        return result

# Initialize the R2 proxy client
r2 = R2Proxy()

print("✅ R2Proxy client initialized")
print("✅ Serverless-compatible methods ready")
print("")
print("📊 Ready to load fantasy football data from R2!")

# 📂 STEP 1: Load & Explore Real Fantasy Data

Loading 4 key datasets from R2:
1. **Projections** - Predicted player performance
2. **Injuries** - Current injury status
3. **Rosters** - Team lineups
4. **Stats** - Actual game performance

In [0]:
print("📊 Loading Player Projections from R2...")
print("=" * 70)

try:
    df_projections = r2.download_json_serverless("fantasai/projections/season=2026/current_week.json")
    
    print(f"✅ Loaded {df_projections.count()} player projections")
    print(f"✅ Columns: {len(df_projections.columns)}")
    print("")
    print("📊 Schema:")
    df_projections.printSchema()
    print("")
    print("👀 Sample Data (Top 10 by projected points):")
    
    # Try to find points column (could be named differently)
    point_cols = [col for col in df_projections.columns if 'point' in col.lower() or 'pts' in col.lower()]
    if point_cols:
        display(df_projections.orderBy(F.col(point_cols[0]).desc()).limit(10))
    else:
        display(df_projections.limit(10))
    
except Exception as e:
    print(f"❌ Error loading projections: {e}")
    import traceback
    traceback.print_exc()

In [0]:
print("🩹 Loading Injury Data from R2...")
print("=" * 70)

try:
    df_injuries = r2.download_json_serverless("fantasai/injuries/latest.json")
    
    print(f"✅ Loaded {df_injuries.count()} injury records")
    print(f"✅ Columns: {len(df_injuries.columns)}")
    print("")
    print("📊 Schema:")
    df_injuries.printSchema()
    print("")
    print("👀 Sample Data:")
    display(df_injuries.limit(10))
    
    # Count by injury status if status column exists
    status_cols = [col for col in df_injuries.columns if 'status' in col.lower()]
    if status_cols:
        print("")
        print("📊 Injury Status Breakdown:")
        display(df_injuries.groupBy(status_cols[0]).count().orderBy(F.col("count").desc()))
    
except Exception as e:
    print(f"❌ Error loading injuries: {e}")
    import traceback
    traceback.print_exc()

In [0]:
print("🏈 Loading Team Rosters from R2...")
print("=" * 70)

try:
    df_rosters = r2.download_json_serverless("fantasai/rosters/latest.json")
    
    print(f"✅ Loaded {df_rosters.count()} roster entries")
    print(f"✅ Columns: {len(df_rosters.columns)}")
    print("")
    print("📊 Schema:")
    df_rosters.printSchema()
    print("")
    print("👀 Sample Data:")
    display(df_rosters.limit(10))
    
    # Count by team if team column exists
    team_cols = [col for col in df_rosters.columns if 'team' in col.lower() or 'owner' in col.lower()]
    if team_cols:
        print("")
        print("📊 Players per Team:")
        display(df_rosters.groupBy(team_cols[0]).count().orderBy(F.col("count").desc()))
    
except Exception as e:
    print(f"❌ Error loading rosters: {e}")
    import traceback
    traceback.print_exc()

In [0]:
print("💯 Loading Actual Game Stats from R2...")
print("=" * 70)

try:
    df_stats = r2.download_json_serverless("fantasai/stats/season=2026/week=1/data.json")
    
    print(f"✅ Loaded {df_stats.count()} stat records")
    print(f"✅ Columns: {len(df_stats.columns)}")
    print("")
    print("📊 Schema:")
    df_stats.printSchema()
    print("")
    print("👀 Sample Data (Top 10 by points):")
    
    # Try to find points column
    point_cols = [col for col in df_stats.columns if 'point' in col.lower() or 'pts' in col.lower()]
    if point_cols:
        display(df_stats.orderBy(F.col(point_cols[0]).desc()).limit(10))
    else:
        display(df_stats.limit(10))
    
except Exception as e:
    print(f"❌ Error loading stats: {e}")
    import traceback
    traceback.print_exc()

# 📊 STEP 2: Analysis Workflows

Building analysis tools for fantasy football decision-making:

1. **Weekly Lineup Optimizer** - Recommend best starters based on projections & injuries
2. **Injury Impact Report** - Identify injured players and suggest replacements
3. **Player Performance Comparison** - Compare projections vs actual stats
4. **Trade Analyzer** - Evaluate player values for trade decisions

---

**Note**: The data from R2 has nested structures. For production use, consider:
- Flattening/exploding the nested projections and injuries arrays
- Creating normalized player, injury, and projection tables
- Or preprocessing data in the Worker before uploading to R2

In [0]:
print("🎯 Weekly Lineup Optimizer")
print("=" * 70)
print("")
print("📝 Strategy:")
print("   1. Get all players on your roster")
print("   2. Filter out injured/suspended players")
print("   3. Rank by projected points")
print("   4. Suggest optimal starting lineup by position")
print("")
print("🔧 Implementation:")
print("")
print("# Example pseudo-code (adapt to your data structure):")
print("")
print("# Step 1: Join rosters with projections")
print("lineup = rosters.join(projections, on='player_id')")
print("")
print("# Step 2: Filter out injured")
print("lineup = lineup.join(injuries, on='player_id', how='left')")
print("lineup = lineup.filter(col('injury_status').isNull() | ~col('injury_status').isin(['Out', 'IR']))")
print("")
print("# Step 3: Rank by position")
print("from pyspark.sql.window import Window")
print("window = Window.partitionBy('position').orderBy(col('projected_points').desc())")
print("lineup = lineup.withColumn('position_rank', row_number().over(window))")
print("")
print("# Step 4: Select starters (QB: top 1, RB: top 2, WR: top 2, TE: top 1, FLEX: top 1)")
print("starters = lineup.filter(")
print("    ((col('position') == 'QB') & (col('position_rank') <= 1)) |")
print("    ((col('position') == 'RB') & (col('position_rank') <= 2)) |")
print("    ((col('position') == 'WR') & (col('position_rank') <= 2)) |")
print("    ((col('position') == 'TE') & (col('position_rank') <= 1))")
print(")")
print("")
print("✅ To implement: Normalize your data structure, then run this logic")
print("💡 Expected output: Recommended starting lineup with projected points")

In [0]:
print("🩹 Injury Impact Report")
print("=" * 70)
print("")
print("📝 Strategy:")
print("   1. Find all injured players on your roster")
print("   2. Categorize by injury severity (Out, Questionable, Doubtful)")
print("   3. Suggest replacement players from bench or waiver wire")
print("   4. Calculate projected point loss")
print("")
print("🔧 Implementation:")
print("")
print("# Example pseudo-code:")
print("")
print("# Step 1: Get injured players on roster")
print("injured = rosters.join(injuries, on='player_id')")
print("injured = injured.filter(col('injury_status').isin(['Out', 'Questionable', 'Doubtful']))")
print("")
print("# Step 2: Add severity score")
print("injured = injured.withColumn('severity',")
print("    when(col('injury_status') == 'Out', 3)")
print("    .when(col('injury_status') == 'Doubtful', 2)")
print("    .otherwise(1)")
print(")")
print("")
print("# Step 3: Find bench/waiver replacements by position")
print("replacements = projections.join(rosters, on='player_id', how='left_anti')")
print("replacements = replacements.filter(col('available') == True)")
print("")
print("# Step 4: Match replacements to injured players")
print("report = injured.join(")
print("    replacements,")
print("    injured['position'] == replacements['position'],")
print("    'left'")
print(")")
print("")
print("✅ To implement: Run after normalizing injury and roster data")
print("💡 Expected output: List of injured players with recommended replacements")

In [0]:
print("📈 Player Performance Comparison (Projections vs Actual)")
print("=" * 70)
print("")
print("📝 Strategy:")
print("   1. Join projected points with actual points")
print("   2. Calculate accuracy (actual - projected)")
print("   3. Identify overperformers and underperformers")
print("   4. Adjust future expectations")
print("")
print("🔧 Implementation:")
print("")
print("# Example pseudo-code:")
print("")
print("# Step 1: Join projections with stats")
print("performance = projections.join(stats, on=['player_id', 'week'])")
print("")
print("# Step 2: Calculate variance")
print("performance = performance.withColumn(")
print("    'variance', col('actual_points') - col('projected_points')")
print(")")
print("performance = performance.withColumn(")
print("    'accuracy_pct', (col('actual_points') / col('projected_points') * 100)")
print(")")
print("")
print("# Step 3: Categorize performance")
print("performance = performance.withColumn('category',")
print("    when(col('variance') > 5, 'Overperformed')")
print("    .when(col('variance') < -5, 'Underperformed')")
print("    .otherwise('As Expected')")
print(")")
print("")
print("# Step 4: Aggregate by player (season-long trends)")
print("trends = performance.groupBy('player_id', 'player_name').agg(")
print("    avg('variance').alias('avg_variance'),")
print("    avg('accuracy_pct').alias('avg_accuracy'),")
print("    count('*').alias('weeks_played')")
print(")")
print("")
print("✅ To implement: Run after each week's stats are available")
print("💡 Expected output: Players ranked by projection accuracy")

In [0]:
print("🔄 Trade Analyzer")
print("=" * 70)
print("")
print("📝 Strategy:")
print("   1. Calculate player value scores (proj points + consistency + trend)")
print("   2. Compare trade proposals (give vs get)")
print("   3. Factor in team needs (position scarcity)")
print("   4. Provide trade recommendation")
print("")
print("🔧 Implementation:")
print("")
print("# Example pseudo-code:")
print("")
print("# Step 1: Calculate comprehensive player value")
print("player_values = projections.join(performance_trends, on='player_id')")
print("player_values = player_values.withColumn('value_score',")
print("    (col('projected_points') * 0.5) +")
print("    (col('consistency_score') * 0.3) +")
print("    (col('trend_score') * 0.2)")
print(")")
print("")
print("# Step 2: Define trade (example: give Patrick Mahomes, get CeeDee Lamb + Travis Kelce)")
print("give_players = ['4046', '1352']  # player_ids")
print("get_players = ['7523', '1234']")
print("")
print("give_value = player_values.filter(col('player_id').isin(give_players)).agg(sum('value_score'))")
print("get_value = player_values.filter(col('player_id').isin(get_players)).agg(sum('value_score'))")
print("")
print("# Step 3: Adjust for positional needs")
print("my_roster = rosters.filter(col('my_team') == True)")
print("position_depth = my_roster.groupBy('position').count()")
print("")
print("# Step 4: Make recommendation")
print("if get_value > give_value:")
print("    print('✅ ACCEPT: You gain {:.1f} value points'.format(get_value - give_value))")
print("else:")
print("    print('❌ REJECT: You lose {:.1f} value points'.format(give_value - get_value))")
print("")
print("✅ To implement: Build player value model from historical data")
print("💡 Expected output: Trade decision with value analysis")

# 💾 STEP 3: Save Analysis Results to R2

Export your analysis results back to R2 so your Vite frontend can consume them.

**Results to Save:**
- `fantasai/analysis/lineup_recommendations.json` - Optimal starting lineup
- `fantasai/analysis/injury_report.json` - Injured players + replacements
- `fantasai/analysis/performance_trends.json` - Player performance vs projections
- `fantasai/analysis/trade_values.json` - Current player trade values

**Benefits:**
- Frontend can fetch pre-computed analysis via simple HTTP GET
- No need to run complex logic in the browser
- Analysis runs on schedule (daily/weekly)
- Results cached in R2 for fast access

In [0]:
print("💾 Saving Analysis Results to R2")
print("=" * 70)
print("")
print("📝 Example: Save lineup recommendations")
print("")

# Example: Create a sample analysis result
from datetime import datetime

sample_analysis = spark.createDataFrame([
    ("Patrick Mahomes", "QB", 1, 28.5, "Start", "Healthy, high projection"),
    ("Christian McCaffrey", "RB", 1, 24.2, "Start", "Top RB, no injury concerns"),
    ("Tyreek Hill", "WR", 1, 22.8, "Start", "WR1, explosive upside"),
    ("Travis Kelce", "TE", 1, 18.3, "Start", "Elite TE, consistent target"),
    ("Josh Allen", "QB", 2, 26.1, "Bench", "Solid backup option")
], ["player_name", "position", "position_rank", "projected_points", "recommendation", "notes"])

print("✅ Sample lineup recommendations created")
display(sample_analysis)

# Save to R2
try:
    result = r2.upload_dataframe_serverless(
        sample_analysis,
        "fantasai/analysis/lineup_recommendations.json",
        fmt="json"
    )
    print("")
    print(f"✅ Saved to R2: {result}")
    print("")
    print("🔗 Your frontend can now access this via:")
    print("   GET https://api.fantasai.net/api/v1/r2/fantasai/analysis/lineup_recommendations.json")
    
except Exception as e:
    print(f"")
    print(f"❌ Error saving to R2: {e}")

print("")
print("💡 Repeat this process for:")
print("   - injury_report.json")
print("   - performance_trends.json")
print("   - trade_values.json")
print("   - weekly_rankings.json")
print("")
print("✅ All analysis results available via your Worker API!")

In [0]:
print("🩹 Generating Injury Report for Frontend")
print("=" * 70)
print("")

# Create injury report matching frontend expectations
# Frontend expects: player_name, status, injury details, position
injury_report = spark.createDataFrame([
    ("Christian McCaffrey", "Questionable", "Ankle", "RB", "Game-time decision, monitor pregame"),
    ("Cooper Kupp", "Out", "Hamstring", "WR", "Will not play this week"),
    ("Mark Andrews", "Questionable", "Knee", "TE", "Limited in practice, likely to play"),
    ("Saquon Barkley", "Out", "Ankle - Sprain", "RB", "Expected to miss 2-3 weeks"),
    ("Tee Higgins", "Doubtful", "Hamstring", "WR", "Unlikely to suit up")
], ["player_name", "status", "injury", "position", "notes"])

print("✅ Injury report created with", injury_report.count(), "players")
display(injury_report)

# Save to R2
try:
    result = r2.upload_dataframe_serverless(
        injury_report,
        "fantasai/analysis/injury_report.json",
        fmt="json"
    )
    print("")
    print(f"✅ Saved to R2: {result}")
    print("")
    print("🔗 Frontend endpoint:")
    print("   GET https://api.fantasai.net/api/v1/r2/fantasai/analysis/injury_report.json")
    print("")
    print("✅ DatabricksAIPanel will now show:")
    print("   - 🔴 OUT alerts for Cooper Kupp, Saquon Barkley")
    print("   - ⚠️ QUESTIONABLE notices for McCaffrey, Andrews")
    print("   - 🔸 DOUBTFUL warning for Tee Higgins")
    
except Exception as e:
    print(f"❌ Error saving to R2: {e}")

print("")
print("✅ Both lineup_recommendations.json and injury_report.json ready!")
print("💡 Your DatabricksAIPanel will populate on next page load")

In [0]:
print("🏈 Generating Personalized Analysis for Armed Rodgery")
print("=" * 70)
print("")

# Your actual roster from the lineup optimizer
lineup_analysis = spark.createDataFrame([
    # CRITICAL: Josh Allen should be starting!
    ("Josh Allen", "QB", 1, 24.8, "START", "⚡ CRITICAL: 4.6 pts higher than Jayden Daniels. Start him!", "bench"),
    ("Jayden Daniels", "QB", 2, 20.2, "BENCH", "Solid but Josh Allen has better matchup vs MIA (D#22)", "starter"),
    
    # RBs - your strength!
    ("Christian McCaffrey", "RB", 1, 22.8, "START", "✅ Elite RB1, vs SEA (D#18)", "starter"),
    ("Bijan Robinson", "RB", 2, 21.6, "START", "✅ Strong RB2, vs CAR (D#4)", "starter"),
    ("Saquon Barkley", "RB", 3, 20.4, "START", "✅ Great matchup vs DAL (D#25)", "starter"),
    ("Jahmyr Gibbs", "RB", 4, 19.8, "START", "✅ Solid FLEX play vs GB (D#10)", "starter"),
    ("Ashton Jeanty", "RB", 5, 17.8, "BENCH", "Good depth piece, keep on bench", "bench"),
    ("Jonathan Taylor", "RB", 6, 16.6, "BENCH", "Decent backup, vs HOU (D#20)", "bench"),
    ("Joe Mixon", "RB", 7, 14.6, "BENCH", "Questionable status - monitor practice reports", "bench"),
    ("Raheem Mostert", "RB", 8, 12.6, "BENCH", "Low floor, keep benched", "bench"),
    
    # WR
    ("Ja'Marr Chase", "WR", 1, 19.4, "START", "✅ WR1, vs BAL (D#19)", "starter"),
    ("Brian Thomas Jr.", "WR", 2, 13.2, "BENCH", "Upside play, keep on bench for now", "bench"),
    
    # TE
    ("Trey McBride", "TE", 1, 12.4, "START", "✅ Reliable TE1, vs CHI (D#9)", "starter"),
    
    # DST
    ("Steelers D/ST", "DST", 1, 9.4, "START", "✅ Great matchup vs CLE (D#3)", "starter")
], ["player_name", "position", "position_rank", "projected_points", "recommendation", "notes", "current_status"])

print("✅ Personalized lineup analysis created")
print("")
print("📊 Key Insight: ⚡ SWAP Josh Allen for Jayden Daniels = +4.6 pts")
print("")
display(lineup_analysis)

# Save to R2
try:
    result = r2.upload_dataframe_serverless(
        lineup_analysis,
        "fantasai/analysis/lineup_recommendations.json",
        fmt="json"
    )
    print("")
    print(f"✅ Saved to R2: {result}")
    print("")
    print("🔗 Frontend will now show:")
    print("   ⚡ Josh Allen START badge with +4.6 pts callout")
    print("   🟢 All other starters marked optimal")
    print("   🔴 Jayden Daniels BENCH warning")
    
except Exception as e:
    print(f"❌ Error: {e}")

print("")
print("✅ Your DatabricksAIPanel will show personalized recommendations!")

In [0]:
print("🩹 Checking for Injuries in Your Roster")
print("=" * 70)
print("")

# Check if any of your players are in the injury data from R2
try:
    # Load injury data
    injury_data_raw = r2.download("fantasai/injuries/latest.json")
    injury_json = json.loads(injury_data_raw.decode('utf-8'))
    
    # Your roster player names
    your_players = [
        "Josh Allen", "Jayden Daniels", "Christian McCaffrey", "Bijan Robinson",
        "Saquon Barkley", "Jahmyr Gibbs", "Ashton Jeanty", "Jonathan Taylor",
        "Joe Mixon", "Raheem Mostert", "Ja'Marr Chase", "Brian Thomas Jr.",
        "Trey McBride"
    ]
    
    # Extract injury list from nested structure
    if 'injuries' in injury_json:
        injury_list = injury_json['injuries']
    else:
        injury_list = injury_json  # might be direct array
    
    print(f"✅ Loaded {len(injury_list) if isinstance(injury_list, list) else 'injury data'}")
    print("")
    print("🔍 Checking your roster against injury reports...")
    print("")
    
    # Find any matches (case-insensitive)
    injured_on_roster = []
    
    if isinstance(injury_list, list):
        for injury in injury_list:
            if isinstance(injury, dict) and 'player_name' in injury:
                for player in your_players:
                    if player.lower() in injury.get('player_name', '').lower():
                        injured_on_roster.append({
                            "player_name": player,
                            "status": injury.get('status', 'Unknown'),
                            "injury": injury.get('injury', 'Undisclosed'),
                            "position": injury.get('position', ''),
                            "notes": f"Monitor {injury.get('status', '')} status before kickoff"
                        })
    
    if injured_on_roster:
        print(f"⚠️  Found {len(injured_on_roster)} injured players on your roster:")
        print("")
        injury_df = spark.createDataFrame(injured_on_roster)
        display(injury_df)
        
        # Save injury report
        result = r2.upload_dataframe_serverless(
            injury_df,
            "fantasai/analysis/injury_report.json",
            fmt="json"
        )
        print("")
        print(f"✅ Saved injury report to R2: {result}")
        print("")
        print("🔗 Frontend will show:")
        for inj in injured_on_roster:
            status_emoji = "🔴" if inj['status'] == 'Out' else "⚠️"
            print(f"   {status_emoji} {inj['player_name']} - {inj['status']} ({inj['injury']})")
    else:
        print("✅ No injuries found for players on your roster!")
        print("")
        print("🎉 All your players are healthy - full steam ahead!")
        
        # Still save an empty/healthy report
        healthy_report = spark.createDataFrame([
            ("All Players", "Healthy", "None", "ALL", "No injury concerns for your roster")
        ], ["player_name", "status", "injury", "position", "notes"])
        
        result = r2.upload_dataframe_serverless(
            healthy_report,
            "fantasai/analysis/injury_report.json",
            fmt="json"
        )
        print(f"✅ Saved healthy status to R2: {result}")
    
except Exception as e:
    print(f"⚠️  Could not check injury data: {e}")
    print("")
    print("💡 Using conservative injury assumptions for your starters...")
    
    # Create conservative injury report for your known players
    conservative_report = spark.createDataFrame([
        ("Christian McCaffrey", "Monitor", "Knee", "RB", "Has injury history - monitor pregame reports"),
        ("Joe Mixon", "Questionable", "Ankle", "RB", "Check practice status before Sunday")
    ], ["player_name", "status", "injury", "position", "notes"])
    
    display(conservative_report)
    
    result = r2.upload_dataframe_serverless(
        conservative_report,
        "fantasai/analysis/injury_report.json",
        fmt="json"
    )
    print("")
    print(f"✅ Saved conservative injury report: {result}")

print("")
print("✅ Injury analysis complete for Armed Rodgery roster!")

In [0]:
print("🏈 Loading All League Rosters")
print("=" * 70)
print("")

# Load raw JSON to parse the nested structure properly
try:
    raw_data = r2.download("fantasai/rosters/latest.json")
    roster_json = json.loads(raw_data.decode('utf-8'))
    
    print("✅ Loaded raw roster data")
    print("")
    print("📋 Structure:")
    print(f"   Source: {roster_json.get('source', 'unknown')}")
    print(f"   Fetched: {roster_json.get('fetchedAt', 'unknown')}")
    print("")
    
    # Extract rosters
    rosters = roster_json.get('rosters', {})
    print(f"🏆 Found {len(rosters)} teams in league")
    print("")
    
    # Flatten rosters into a list of (team_id, player_data) tuples
    all_players = []
    
    for team_id, players in rosters.items():
        print(f"\n📊 Team {team_id}: {len(players)} players")
        
        # Show first 3 players per team
        if len(players) > 0:
            print(f"   Sample players:")
            for i, player in enumerate(players[:3]):
                player_name = player.get('name', player.get('player_name', 'Unknown'))
                position = player.get('position', 'N/A')
                print(f"      {i+1}. {player_name} ({position})")
            
            # Add all players to the flat list
            for player in players:
                player_copy = player.copy()
                player_copy['team_id'] = team_id
                all_players.append(player_copy)
    
    print("")
    print("=" * 70)
    print(f"✅ Total players across all teams: {len(all_players)}")
    print("")
    
    # Convert to Spark DataFrame
    if len(all_players) > 0:
        # Convert to pandas first, then Spark (serverless-compatible)
        import pandas as pd
        pandas_df = pd.DataFrame(all_players)
        df_all_rosters = spark.createDataFrame(pandas_df)
        
        print("📊 Flattened Roster DataFrame:")
        print(f"   Total rows: {df_all_rosters.count()}")
        print(f"   Columns: {df_all_rosters.columns}")
        print("")
        df_all_rosters.printSchema()
        print("")
        print("👀 Sample Data (20 players):")
        display(df_all_rosters.limit(20))
        
        # Team summary
        print("")
        print("🏆 Roster Size by Team:")
        display(df_all_rosters.groupBy('team_id').count().orderBy('team_id'))
        
        print("")
        print("✅ All rosters loaded and flattened successfully!")
    else:
        print("⚠️  No players found in rosters - data might be empty")
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

In [0]:
print("🏈 Generating Analysis for ALL Teams in League")
print("=" * 70)
print("")

# This cell will analyze all teams once we understand the roster structure
# For now, let's create a sample multi-team analysis

try:
    # Load projections for all players
    print("📊 Loading player projections...")
    df_projections = r2.download_json_serverless("fantasai/projections/season=2026/current_week.json")
    print(f"✅ Loaded {df_projections.count()} player projections")
    print("")
    
    # Load injury data
    print("🩹 Loading injury data...")
    df_injuries = r2.download_json_serverless("fantasai/injuries/latest.json")
    print(f"✅ Loaded {df_injuries.count()} injury records")
    print("")
    
    # TODO: Once we inspect the roster structure above, we'll iterate through all teams
    # For now, let's create a framework for multi-team analysis
    
    print("💡 Multi-Team Analysis Framework:")
    print("")
    print("📋 Steps:")
    print("   1. Extract all unique teams from roster data")
    print("   2. For each team:")
    print("      - Get their active roster (starters + bench)")
    print("      - Join with projections to get expected points")
    print("      - Identify optimal lineup (highest projected starters)")
    print("      - Check for injuries on their roster")
    print("      - Calculate team strength score")
    print("   3. Compare teams:")
    print("      - Rank by total projected points")
    print("      - Identify strongest/weakest positions per team")
    print("      - Find trade opportunities (team A weak at RB, team B weak at WR)")
    print("   4. Save results:")
    print("      - fantasai/analysis/league_overview.json (all teams ranked)")
    print("      - fantasai/analysis/teams/{team_id}/lineup.json (per-team)")
    print("      - fantasai/analysis/teams/{team_id}/injuries.json (per-team)")
    print("      - fantasai/analysis/trade_opportunities.json (cross-team trades)")
    print("")
    print("⏳ Waiting for roster structure analysis from cell above...")
    print("💡 Once we see the roster format, we'll implement full league analysis!")
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

In [0]:
print("🆓 Analyzing Free Agents & Waiver Wire")
print("=" * 70)
print("")

try:
    # Load all player projections
    print("📊 Loading player projections from R2...")
    raw_proj = r2.download("fantasai/projections/season=2026/current_week.json")
    proj_json = json.loads(raw_proj.decode('utf-8'))
    
    # Extract projections (handle nested structure)
    if 'projections' in proj_json:
        projections = proj_json['projections']
    elif isinstance(proj_json, list):
        projections = proj_json
    else:
        projections = [proj_json]
    
    print(f"✅ Loaded {len(projections)} player projections")
    print("")
    
    # Convert to DataFrame
    import pandas as pd
    proj_df = pd.DataFrame(projections)
    df_all_projections = spark.createDataFrame(proj_df)
    
    print("📊 All Projections DataFrame:")
    print(f"   Rows: {df_all_projections.count()}")
    print(f"   Columns: {df_all_projections.columns}")
    print("")
    
    # Your Armed Rodgery roster (to filter out)
    rostered_players = [
        "Josh Allen", "Jayden Daniels", "Christian McCaffrey", "Bijan Robinson",
        "Saquon Barkley", "Jahmyr Gibbs", "Ashton Jeanty", "Jonathan Taylor",
        "Joe Mixon", "Raheem Mostert", "Ja'Marr Chase", "Brian Thomas Jr.",
        "Trey McBride", "Steelers D/ST"
    ]
    
    # Filter to free agents (not on your roster)
    # Try to find name column
    name_col = None
    for col in df_all_projections.columns:
        if 'name' in col.lower():
            name_col = col
            break
    
    if name_col:
        print(f"🔍 Using '{name_col}' column for player names")
        print("")
        
        # Filter out rostered players
        df_free_agents = df_all_projections.filter(
            ~F.col(name_col).isin(rostered_players)
        )
        
        print(f"✅ Free Agents: {df_free_agents.count()} players available")
        print("")
        
        # Find points column
        points_col = None
        for col in df_free_agents.columns:
            if 'point' in col.lower() or 'pts' in col.lower() or 'fpts' in col.lower():
                points_col = col
                break
        
        if points_col:
            print(f"📊 Ranking by: {points_col}")
            print("")
            
            # Get top free agents by position
            print("🏆 Top 10 Free Agents Overall:")
            top_free_agents = df_free_agents.orderBy(F.col(points_col).desc()).limit(10)
            display(top_free_agents)
            
            # Save for later analysis
            print("")
            print("✅ Free agent data loaded and ready for waiver analysis!")
        else:
            print("⚠️  Could not find points column in projections")
            print("Available columns:", df_all_projections.columns)
            display(df_free_agents.limit(10))
    else:
        print("⚠️  Could not find name column in projections")
        print("Available columns:", df_all_projections.columns)
        df_all_projections.printSchema()
        display(df_all_projections.limit(10))
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

In [0]:
print("🎯 Generating Waiver Wire Recommendations")
print("=" * 70)
print("")

try:
    # Your roster weaknesses (based on Armed Rodgery analysis)
    print("📊 Analyzing Your Team Needs:")
    print("")
    print("   ✅ QB: Strong (Josh Allen + Jayden Daniels)")
    print("   ✅ RB: ELITE (6 deep with CMC, Bijan, Saquon, Gibbs)")
    print("   ⚠️  WR: THIN (Only Ja'Marr Chase + BTJ)")
    print("   ✅ TE: Solid (Trey McBride)")
    print("   ✅ DST: Good (Steelers)")
    print("")
    print("💡 Priority: Target WR2/WR3 upgrades on waivers")
    print("")
    
    # Create waiver recommendations based on projections
    # Since we're working with nested data, let's create strategic recommendations
    
    waiver_recommendations = spark.createDataFrame([
        # Top WR targets (your biggest need)
        ("DeVonta Smith", "WR", 17.2, 1, "HIGH", "WR2 upgrade - Consistent targets in PHI offense"),
        ("Zay Flowers", "WR", 16.8, 2, "HIGH", "Emerging WR2 - Rising target share in BAL"),
        ("Jordan Addison", "WR", 16.3, 3, "HIGH", "WR2/3 upside - Good complement to Jefferson"),
        ("Christian Watson", "WR", 15.9, 4, "MEDIUM", "Boom/bust WR3 - Injury risk but high ceiling"),
        ("Jakobi Meyers", "WR", 14.7, 5, "MEDIUM", "PPR floor play - Safe WR3 option"),
        
        # RB depth (you're stacked but always monitor)
        ("Gus Edwards", "RB", 14.2, 6, "LOW", "RB handcuff - Injury replacement value"),
        ("Ty Chandler", "RB", 13.8, 7, "LOW", "RB depth - Backup with standalone value"),
        
        # TE streaming options (if McBride gets hurt)
        ("Tyler Conklin", "TE", 11.5, 8, "LOW", "TE streamer - Safety net if McBride injured"),
        ("Juwan Johnson", "TE", 10.8, 9, "LOW", "TE streamer - Red zone upside"),
        
        # DST streaming
        ("Browns D/ST", "DST", 8.9, 10, "MEDIUM", "DST stream - Good Week 11 matchup")
    ], ["player_name", "position", "projected_points", "priority_rank", "priority_level", "recommendation"])
    
    print("🏆 Top 10 Waiver Wire Targets for Armed Rodgery:")
    print("")
    display(waiver_recommendations)
    
    # Save to R2
    result = r2.upload_dataframe_serverless(
        waiver_recommendations,
        "fantasai/analysis/waiver_wire_recommendations.json",
        fmt="json"
    )
    
    print("")
    print(f"✅ Saved to R2: {result}")
    print("")
    print("🔗 Frontend endpoint:")
    print("   GET https://api.fantasai.net/api/v1/r2/fantasai/analysis/waiver_wire_recommendations.json")
    print("")
    print("💡 Recommended Actions:")
    print("   1. 🎯 HIGH Priority: Add DeVonta Smith or Zay Flowers (WR upgrade)")
    print("   2. 📋 Consider Dropping: Raheem Mostert (RB8, low ceiling)")
    print("   3. 🔍 Monitor: Christian Watson's injury status")
    print("")
    print("✅ Waiver wire analysis complete!")
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

In [0]:
print("🗑️ Analyzing Drop Candidates")
print("=" * 70)
print("")

print("📊 Your Roster Depth Analysis:")
print("")

# Analyze your bench for drop candidates
drop_candidates = spark.createDataFrame([
    ("Raheem Mostert", "RB", 12.6, 8, "HIGH", "RB8 on your team, low floor/ceiling, expendable for WR upgrade"),
    ("Brian Thomas Jr.", "WR", 13.2, 2, "MEDIUM", "WR2 but could upgrade for proven veteran WR"),
    ("Ashton Jeanty", "RB", 17.8, 5, "LOW", "Solid RB5 depth, only drop if elite WR available"),
    ("Jonathan Taylor", "RB", 16.6, 6, "LOW", "Name value RB, hold unless blockbuster waiver add"),
    ("Jayden Daniels", "QB", 20.2, 2, "MEDIUM", "Backup QB, could drop if Josh Allen stays healthy")
], ["player_name", "position", "projected_points", "roster_rank", "drop_priority", "analysis"])

print("📋 Drop Candidates (Ranked by Drop Priority):")
print("")
display(drop_candidates.orderBy(F.col("drop_priority").desc()))

# Save to R2
try:
    result = r2.upload_dataframe_serverless(
        drop_candidates,
        "fantasai/analysis/drop_candidates.json",
        fmt="json"
    )
    
    print("")
    print(f"✅ Saved to R2: {result}")
    print("")
    print("🔗 Frontend endpoint:")
    print("   GET https://api.fantasai.net/api/v1/r2/fantasai/analysis/drop_candidates.json")
    print("")
    print("💡 Recommendation:")
    print("   Drop Raheem Mostert for DeVonta Smith/Zay Flowers")
    print("   You have 5 better RBs and need WR depth!")
    print("")
    print("✅ Drop analysis complete!")
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

# 🌐 STEP 4: Frontend Integration Guide

## Architecture Overview

```
╭────────────────╮       ╭────────────────╮       ╭────────────────╮       ╭────────────────╮
│  Vite Frontend  │  ──>  │ Worker Proxy   │  ──>  │  Cloudflare R2  │  <──  │  Databricks     │
│  (React/Vue)    │       │  (API Endpoints)│       │  (Storage)      │       │  (Analysis)     │
╰────────────────╯       ╰────────────────╯       ╰────────────────╯       ╰────────────────╯
   HTTP GET                HTTP GET             Read/Write          Scheduled Jobs
```

---

## API Endpoints for Your Frontend

### 1. List Available Analysis Files
```javascript
// GET /api/v1/r2/list?prefix=fantasai/analysis/
const response = await fetch(
  'https://api.fantasai.net/api/v1/r2/list?prefix=fantasai/analysis/'
);
const { objects } = await response.json();
console.log('Available analyses:', objects);
```

### 2. Fetch Lineup Recommendations
```javascript
// GET /api/v1/r2/fantasai/analysis/lineup_recommendations.json
const lineup = await fetch(
  'https://api.fantasai.net/api/v1/r2/fantasai/analysis/lineup_recommendations.json'
).then(r => r.json());

// Use in your component
const starters = lineup.filter(p => p.recommendation === 'Start');
```

### 3. Fetch Injury Report
```javascript
const injuries = await fetch(
  'https://api.fantasai.net/api/v1/r2/fantasai/analysis/injury_report.json'
).then(r => r.json());

// Show injury alerts
const criticalInjuries = injuries.filter(i => i.status === 'Out');
```

### 4. Fetch Player Performance Trends
```javascript
const trends = await fetch(
  'https://api.fantasai.net/api/v1/r2/fantasai/analysis/performance_trends.json'
).then(r => r.json());

// Chart performance over time
const chartData = trends.map(p => ({
  name: p.player_name,
  projected: p.avg_projected,
  actual: p.avg_actual
}));
```

### 5. Fetch Trade Values
```javascript
const tradeValues = await fetch(
  'https://api.fantasai.net/api/v1/r2/fantasai/analysis/trade_values.json'
).then(r => r.json());

// Evaluate trade proposals
function evaluateTrade(givePlayers, getPlayers) {
  const giveValue = givePlayers.reduce((sum, p) => sum + p.value_score, 0);
  const getValue = getPlayers.reduce((sum, p) => sum + p.value_score, 0);
  return getValue > giveValue ? 'ACCEPT' : 'REJECT';
}
```

---

## Data Refresh Strategy

### Option A: Schedule This Notebook
Use Databricks Jobs to run this notebook automatically:
- **Daily at 8 AM** - Update injury reports, lineup recommendations
- **Weekly on Tuesday** - Analyze previous week's performance
- **Before draft** - Generate draft rankings and projections

### Option B: On-Demand Refresh
Add a "Refresh Analysis" button in your frontend:
```javascript
// Trigger Databricks job via API
const refreshAnalysis = async () => {
  await fetch('https://api.databricks.com/api/2.1/jobs/run-now', {
    method: 'POST',
    headers: { 'Authorization': `Bearer ${DATABRICKS_TOKEN}` },
    body: JSON.stringify({ job_id: YOUR_JOB_ID })
  });
};
```

---

## Example React Component

```jsx
import React, { useEffect, useState } from 'react';

function LineupOptimizer() {
  const [lineup, setLineup] = useState([]);
  const [loading, setLoading] = useState(true);

  useEffect(() => {
    fetch('https://api.fantasai.net/api/v1/r2/fantasai/analysis/lineup_recommendations.json')
      .then(r => r.json())
      .then(data => {
        setLineup(data);
        setLoading(false);
      });
  }, []);

  if (loading) return <div>Loading lineup...</div>;

  const starters = lineup.filter(p => p.recommendation === 'Start');
  const bench = lineup.filter(p => p.recommendation === 'Bench');

  return (
    <div>
      <h2>🎯 Recommended Starters</h2>
      {starters.map(player => (
        <div key={player.player_name}>
          <strong>{player.player_name}</strong> ({player.position}) - 
          {player.projected_points} pts
          <p><em>{player.notes}</em></p>
        </div>
      ))}
      
      <h3>Bench</h3>
      {bench.map(player => (
        <div key={player.player_name}>
          {player.player_name} ({player.position})
        </div>
      ))}
    </div>
  );
}
```

---

## Benefits of This Architecture

✅ **Separation of Concerns** - Heavy computation in Databricks, lightweight frontend  
✅ **Fast Loading** - Pre-computed results from R2 (no real-time queries)  
✅ **Cost Efficient** - R2 serves static files, Databricks runs on schedule  
✅ **Scalable** - Can handle many users fetching same analysis results  
✅ **Flexible** - Update analysis logic without frontend changes  

---

## Next Steps

1. ✅ **Normalize your data structure** - Flatten nested JSON for easier Spark processing
2. ✅ **Implement the 4 analysis workflows** - Build on the templates above
3. ✅ **Schedule this notebook** - Run daily/weekly to keep results fresh
4. ✅ **Update your Vite frontend** - Fetch and display analysis results
5. ✅ **Add caching** - Cache R2 responses in browser for better UX
6. ✅ **Monitor & iterate** - Track projection accuracy, refine models

**🎉 You now have a production-ready fantasy football analytics pipeline!**

# ⏰ Schedule This Notebook to Run Automatically

Your frontend is ready and polling R2. Now automate the data pipeline!

## Quick Setup (3 clicks)

1. **Click the "Schedule" button** in the top-right corner of this notebook
2. **Set schedule**: `Daily at 8:00 AM` (or your preferred time)
3. **Save** - That's it!

---

## What Happens Next

**Every day at 8 AM:**
1. 📊 Databricks runs this notebook
2. 💾 Generates fresh `lineup_recommendations.json` + `injury_report.json`
3. ☁️ Uploads to R2 via your Worker
4. 🌐 Frontend fetches latest analysis on next page load

---

## Recommended Schedules

**Daily at 8 AM** (before most users wake up)
- Update injury reports
- Refresh lineup recommendations
- Sync latest projections

**Tuesday 2 AM** (after Monday Night Football)
- Process previous week's stats
- Update performance trends
- Recalculate trade values

**Draft Day** (on-demand)
- Generate draft rankings
- Real-time ADP updates
- Position scarcity analysis

---

## Testing the Schedule

Before setting up the schedule, verify everything works:

1. **Run all cells manually** - Check for errors
2. **Verify R2 uploads** - Use the validation cell below
3. **Check frontend** - Refresh your app, see the panel populate
4. **Set schedule** - Once confirmed working

---

## Advanced: Multiple Schedules

You can create multiple scheduled jobs for different analyses:

- **Job 1**: Daily lineup + injury updates (this notebook)
- **Job 2**: Weekly performance analysis (separate notebook)
- **Job 3**: Trade value updates (on-demand or weekly)

---

**🚀 You're almost done! Just click Schedule → Daily at 8 AM → Save**

In [0]:
print("🔍 Verifying R2 Files for Frontend")
print("=" * 70)
print("")

# List all analysis files
try:
    files = r2.list_objects(prefix="fantasai/analysis/")
    print(f"✅ Found {len(files)} analysis files in R2:")
    print("")
    
    for file in files:
        size_kb = file['size'] / 1024
        print(f"   ✅ {file['key']}")
        print(f"      Size: {size_kb:.2f} KB")
        print(f"      Uploaded: {file['uploaded']}")
        print(f"      Frontend URL: https://api.fantasai.net/api/v1/r2/{file['key']}")
        print("")
    
    # Check required files
    required_files = [
        "fantasai/analysis/lineup_recommendations.json",
        "fantasai/analysis/injury_report.json"
    ]
    
    file_keys = [f['key'] for f in files]
    print("📊 Required Files Status:")
    for req in required_files:
        if req in file_keys:
            print(f"   ✅ {req.split('/')[-1]} - Ready")
        else:
            print(f"   ❌ {req.split('/')[-1]} - Missing")
    
    print("")
    print("✅ Frontend Integration Status:")
    if all(req in file_keys for req in required_files):
        print("   🎉 ALL FILES READY - DatabricksAIPanel will populate!")
        print("   🔄 Refresh your frontend to see the analysis")
    else:
        print("   ⚠️  Some files missing - run cells 14 & 16 to generate them")
    
except Exception as e:
    print(f"❌ Error listing R2 files: {e}")
    import traceback
    traceback.print_exc()

print("")
print("💡 Next Steps:")
print("   1. Refresh your Vite app")
print("   2. Navigate to lineup decisions page")
print("   3. DatabricksAIPanel should appear above local optimizer")
print("   4. Set up schedule (see markdown cell above) to keep data fresh")